# Student Performance Prediction Using Machine Learning

Compare Logistic Regression, Gaussian Naive Bayes and RBF-SVM on the mathematics subset of the UCI Student Performance dataset.

**Target:** Low (G3 0–9), Medium (10–14), High (15–20). The grade columns `G1`, `G2` and `G3` are excluded from the input features to avoid direct target leakage.

Run the cells from top to bottom. The dataset is downloaded automatically; no manual upload is required.

In [ ]:
import io, zipfile, urllib.request, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')
url = 'https://archive.ics.uci.edu/static/public/320/student+performance.zip'
with urllib.request.urlopen(url) as response:
    archive_bytes = response.read()
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    df = pd.read_csv(archive.open('student-mat.csv'), sep=';')
print(f'Dataset: {df.shape[0]} rows × {df.shape[1]} columns')
display(df.head())

In [ ]:
# Explore the data and create the three-level target
df['Performance'] = pd.cut(df['G3'], bins=[-1, 9, 14, 20], labels=['Low', 'Medium', 'High'])
print('Missing values:', df.isna().sum().sum())
print('Duplicate rows:', df.duplicated().sum())
display(df['Performance'].value_counts().reindex(['Low','Medium','High']).to_frame('Students'))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df, x='G3', bins=21, ax=axes[0]); axes[0].set_title('Final grade distribution')
sns.countplot(df, x='Performance', order=['Low','Medium','High'], ax=axes[1]); axes[1].set_title('Performance categories')
plt.tight_layout(); plt.show()

In [ ]:
# Split the data before fitting preprocessing to prevent leakage
X = df.drop(columns=['G1', 'G2', 'G3', 'Performance'])
y = df['Performance'].astype(str)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
numeric = X.select_dtypes(include='number').columns
categorical = X.select_dtypes(exclude='number').columns
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical)
])
print('Training rows:', len(X_train), '| Test rows:', len(X_test))

In [ ]:
# Train the three classifiers using the same preprocessing pipeline
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
    'Gaussian Naive Bayes': GaussianNB(),
    'Support Vector Machine': SVC(kernel='rbf', C=1.0, random_state=RANDOM_STATE)
}
models = {name: Pipeline([('preprocessor', preprocessor), ('classifier', clf)]) for name, clf in classifiers.items()}
for model in models.values(): model.fit(X_train, y_train)
print('All models trained.')

In [ ]:
# Evaluate on the held-out test set
predictions = {name: model.predict(X_test) for name, model in models.items()}
test_results = pd.DataFrame([{
    'Model': name, 'Accuracy': accuracy_score(y_test, pred),
    'Precision (Macro)': precision_score(y_test, pred, average='macro', zero_division=0),
    'Recall (Macro)': recall_score(y_test, pred, average='macro', zero_division=0),
    'Macro F1': f1_score(y_test, pred, average='macro', zero_division=0)
} for name, pred in predictions.items()]).sort_values('Macro F1', ascending=False)
display(test_results.round(4))
for name, pred in predictions.items():
    print(f'\\n{name}\\n', classification_report(y_test, pred, labels=['Low','Medium','High'], zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=['Low','Medium','High'], cmap='Blues')
    plt.title(f'Confusion Matrix — {name}'); plt.show()

In [ ]:
# Stratified 5-fold cross-validation on the training set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring={'accuracy':'accuracy','macro_f1':'f1_macro'}, n_jobs=-1)
    cv_rows.append({'Model':name, 'Mean Accuracy':scores['test_accuracy'].mean(), 'Accuracy Std':scores['test_accuracy'].std(), 'Mean Macro F1':scores['test_macro_f1'].mean(), 'Macro F1 Std':scores['test_macro_f1'].std()})
cv_results = pd.DataFrame(cv_rows).sort_values('Mean Macro F1', ascending=False).reset_index(drop=True)
display(cv_results.round(4))
best_name = cv_results.loc[0, 'Model']; best_model = models[best_name]
print('Selected model (highest CV Macro F1):', best_name)

In [ ]:
# Permutation importance on the held-out test set
importance = permutation_importance(best_model, X_test, y_test, scoring='f1_macro', n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
importance_results = pd.DataFrame({'Feature':X_test.columns, 'Importance Mean':importance.importances_mean, 'Importance Std':importance.importances_std}).sort_values('Importance Mean', ascending=False)
display(importance_results.head(15).round(4))
top = importance_results.head(15).sort_values('Importance Mean')
plt.figure(figsize=(9,6)); plt.barh(top['Feature'], top['Importance Mean'], xerr=top['Importance Std']); plt.axvline(0, linestyle='--'); plt.xlabel('Decrease in Macro F1 after shuffling'); plt.title(f'Permutation Importance — {best_name}'); plt.tight_layout(); plt.show()

In [ ]:
# Save results and trained models for reuse
test_results.to_csv('test_set_model_comparison.csv', index=False)
cv_results.to_csv('cross_validation_results.csv', index=False)
importance_results.to_csv('feature_importance.csv', index=False)
pd.DataFrame({'Actual':y_test.to_numpy(), **{name:pred for name,pred in predictions.items()}}).to_csv('test_set_predictions.csv', index=False)
df.to_csv('student_performance_dataset.csv', index=False)
joblib.dump(models, 'all_trained_models.joblib')
joblib.dump(best_model, 'best_model.joblib')
metadata = {'python_version':__import__('sys').version.split()[0], 'scikit_learn_version':sklearn.__version__, 'best_model':best_name, 'target_classes':['Low','Medium','High'], 'random_state':RANDOM_STATE}
with open('model_metadata.json','w') as file: json.dump(metadata, file, indent=2)
print('Saved CSV results, dataset, model files and model_metadata.json')
print('Use the same scikit-learn version when loading the saved joblib files.')